# CNN vs Vision Transformer Comparison

**Dataset:** ISIC 2019  
**Task:** Binary skin lesion classification  
**Models:** Convolutional neural network and Vision Transformer architectures  
**Evaluation:** Accuracy, precision, recall, F1-score, AUC and comparative visualisations

This notebook compares CNN and Vision Transformer approaches under a common experimental pipeline. It was developed in Google Colab; dataset and output paths must be adjusted before execution.

In [ ]:


!pip install -q timm



import os
import glob
import copy
import shutil
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)



if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")



SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True


MAX_PER_CLASS = 1500



OUTPUT_DIR = "/content/drive/MyDrive/ISIC2019_CNN_VS_VIT_COMPARISON"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOCAL_DATA_DIR = "/content/isic2019_cnn_vs_vit_fast"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Output dir:", OUTPUT_DIR)



def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)



required_folders = {"AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"}

def find_isic2019_root(start_dir="/content/drive/MyDrive"):
    candidates = []

    for root, dirs, files in os.walk(start_dir):
        if required_folders.issubset(set(dirs)):
            candidates.append(root)

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Δεν βρέθηκε φάκελος που να περιέχει AK, BCC, BKL, DF, MEL, NV, SCC, VASC."
        )

    print("\nCandidate ISIC2019 folders:")
    for i, c in enumerate(candidates):
        print(i, ":", c)

    return candidates[0]

DATA_DIR = find_isic2019_root("/content/drive/MyDrive")

print("\nUsing DATA_DIR:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR)[:20])



all_classes = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]

suspicious_classes = ["AK", "BCC", "MEL", "SCC"]
benign_like_classes = ["NV", "BKL", "DF", "VASC"]

binary_map = {}
for c in benign_like_classes:
    binary_map[c] = 0

for c in suspicious_classes:
    binary_map[c] = 1

binary_name_map = {
    0: "benign_like",
    1: "suspicious"
}

known_bad_files = [
    "ISIC_0013092_downsampled.jpg",
    "ISIC_0026625.jpg"
]

rows = []

for cls in all_classes:
    cls_dir = os.path.join(DATA_DIR, cls)

    if not os.path.exists(cls_dir):
        raise FileNotFoundError(f"Δεν βρέθηκε ο φάκελος: {cls_dir}")

    paths = []

    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        paths.extend(glob.glob(os.path.join(cls_dir, "**", ext), recursive=True))

    print(cls, "images:", len(paths))

    for p in paths:
        fname = os.path.basename(p)

        if fname in known_bad_files:
            continue

        rows.append({
            "image_path": p,
            "filename": fname,
            "original_class": cls,
            "binary_label": binary_map[cls],
            "binary_class": binary_name_map[binary_map[cls]],
            "image_id": os.path.splitext(fname)[0]
        })

df = pd.DataFrame(rows)

print("\nFull dataframe:", len(df))

print("\nOriginal class distribution:")
print(df["original_class"].value_counts())

print("\nBinary distribution before balancing:")
print(df["binary_class"].value_counts())


df_benign = df[df["binary_label"] == 0].sample(
    n=min(MAX_PER_CLASS, len(df[df["binary_label"] == 0])),
    random_state=SEED
)

df_suspicious = df[df["binary_label"] == 1].sample(
    n=min(MAX_PER_CLASS, len(df[df["binary_label"] == 1])),
    random_state=SEED
)

df = pd.concat([df_benign, df_suspicious], axis=0)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nBalanced subset:")
print(df["binary_class"].value_counts())
print("Total:", len(df))



print("\nCopying selected images to local runtime...")

local_paths = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    src = row["image_path"]
    cls = row["binary_class"]
    fname = row["filename"]

    dst_dir = os.path.join(LOCAL_DATA_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    dst = os.path.join(dst_dir, fname)

    if not os.path.exists(dst):
        shutil.copy2(src, dst)

    local_paths.append(dst)

df["image_path"] = local_paths

print("\nLocal copy completed.")
print(df["binary_class"].value_counts())



train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["binary_label"],
    random_state=SEED
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1765,
    stratify=train_val_df["binary_label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["binary_class"].value_counts())

print("\nValidation distribution:")
print(val_df["binary_class"].value_counts())

print("\nTest distribution:")
print(test_df["binary_class"].value_counts())


train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)



def safe_open_rgb(path):
    try:
        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError("cv2.imread returned None")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img)

    except Exception as e:
        print("Image error:", path, e)
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return Image.fromarray(arr)

train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.82, 1.0),
        ratio=(0.90, 1.10)
    ),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=15),
    T.ColorJitter(
        brightness=0.10,
        contrast=0.12,
        saturation=0.08,
        hue=0.02
    ),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

class ISICBinaryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = safe_open_rgb(row["image_path"])
        label = float(row["binary_label"])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

train_ds = ISICBinaryDataset(train_df, transform=train_tf)
val_ds = ISICBinaryDataset(val_df, transform=eval_tf)
test_ds = ISICBinaryDataset(test_df, transform=eval_tf)



train_labels = train_df["binary_label"].values.astype(int)

class_counts = np.bincount(train_labels, minlength=2)
class_counts = np.maximum(class_counts, 1)

sample_weights = (1.0 / class_counts)[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)



model_configs = [
    {
        "model_key": "efficientnet_b0",
        "model_name": "efficientnet_b0",
        "family": "CNN"
    },
    {
        "model_key": "rexnet_100",
        "model_name": "rexnet_100",
        "family": "CNN"
    },
    {
        "model_key": "convnext_tiny",
        "model_name": "convnext_tiny",
        "family": "CNN-like"
    },
    {
        "model_key": "swin_tiny",
        "model_name": "swin_tiny_patch4_window7_224",
        "family": "Vision Transformer"
    },
    {
        "model_key": "deit_small",
        "model_name": "deit_small_patch16_224",
        "family": "Vision Transformer"
    },
    {
        "model_key": "vit_tiny",
        "model_name": "vit_tiny_patch16_224",
        "family": "Vision Transformer"
    }
]



def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "pred": y_pred
    }

def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 91)

    best_threshold = 0.5
    best_score = -1

    for th in thresholds:
        m = evaluate_at_threshold(y_true, y_prob, th)
        score = m[metric]

        if score > best_score:
            best_score = score
            best_threshold = th

    return best_threshold, best_score

@torch.no_grad()
def get_probs_labels(model, loader):
    model.eval()

    labels_all = []
    probs_all = []

    for images, labels in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(device)

        logits = model(images).view(-1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        probs_all.extend(probs)
        labels_all.extend(labels.numpy())

    return np.array(labels_all).astype(int), np.array(probs_all)


def train_one_model(model_key, model_name, family):
    print("\n" + "=" * 90)
    print(f"Training model: {model_key} | {model_name} | {family}")
    print("=" * 90)

    model_dir = os.path.join(OUTPUT_DIR, model_key)
    os.makedirs(model_dir, exist_ok=True)

    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=1
    ).to(device)

    pos = int(train_df["binary_label"].sum())
    neg = int((train_df["binary_label"] == 0).sum())

    pos_weight = torch.tensor(
        [neg / max(pos, 1)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(USE_AMP and device.type == "cuda")
    )

    best_val_auc = -1
    best_state = None
    best_epoch = -1
    best_threshold = 0.5

    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        running_loss = 0.0
        n = 0

        loop = tqdm(
            train_loader,
            desc=f"{model_key} Epoch {epoch}/{EPOCHS}"
        )

        for images, labels in loop:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                logits = model(images).view(-1)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            n += images.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()

        train_loss = running_loss / max(n, 1)

        y_val, p_val = get_probs_labels(model, val_loader)
        threshold, _ = find_best_threshold(y_val, p_val, metric="f1")
        val_metrics = evaluate_at_threshold(y_val, p_val, threshold)

        print(
            f"{model_key} | Epoch {epoch}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_precision={val_metrics['precision']:.4f} | "
            f"val_recall={val_metrics['recall']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['auc']:.4f} | "
            f"thr={threshold:.2f}"
        )

        history.append({
            "model_key": model_key,
            "model_name": model_name,
            "family": family,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
            "threshold": threshold
        })

        if not np.isnan(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_threshold = threshold

            ckpt_path = os.path.join(model_dir, f"best_{model_key}.pth")

            torch.save({
                "model_key": model_key,
                "model_name": model_name,
                "family": family,
                "model_state": best_state,
                "best_epoch": best_epoch,
                "best_val_auc": best_val_auc,
                "threshold": best_threshold
            }, ckpt_path)

            print("Saved best model:", ckpt_path)

    history_df = pd.DataFrame(history)
    history_path = os.path.join(model_dir, f"{model_key}_history.csv")
    history_df.to_csv(history_path, index=False)

    if best_state is None:
        raise RuntimeError(f"No best state saved for {model_key}")

    model.load_state_dict(best_state)
    model.eval()

    y_test, p_test = get_probs_labels(model, test_loader)
    test_metrics = evaluate_at_threshold(y_test, p_test, best_threshold)

    print("\n=== TEST RESULTS:", model_key, "===")
    print("Best epoch:", best_epoch)
    print("Threshold:", best_threshold)
    print("Accuracy:", test_metrics["accuracy"])
    print("Precision:", test_metrics["precision"])
    print("Recall:", test_metrics["recall"])
    print("F1:", test_metrics["f1"])
    print("AUC:", test_metrics["auc"])

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            test_metrics["pred"],
            target_names=["benign_like", "suspicious"],
            zero_division=0
        )
    )

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, test_metrics["pred"])
    print(cm)

    predictions_df = test_df.copy().reset_index(drop=True)
    predictions_df["true_label"] = y_test
    predictions_df["prob_suspicious"] = p_test
    predictions_df["pred_label"] = test_metrics["pred"]
    predictions_df["true_class"] = predictions_df["true_label"].map(binary_name_map)
    predictions_df["pred_class"] = predictions_df["pred_label"].map(binary_name_map)
    predictions_df["correct"] = predictions_df["true_label"] == predictions_df["pred_label"]

    predictions_path = os.path.join(model_dir, f"{model_key}_test_predictions.csv")
    predictions_df.to_csv(predictions_path, index=False)

    # Confusion matrix figure
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix - {model_key}")
    plt.colorbar()

    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["benign_like", "suspicious"])
    plt.yticks(tick_marks, ["benign_like", "suspicious"])

    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()

    cm_path = os.path.join(model_dir, f"{model_key}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.show()

    result_row = {
        "model_key": model_key,
        "model_name": model_name,
        "family": family,
        "max_per_class": MAX_PER_CLASS,
        "epochs": EPOCHS,
        "best_epoch": best_epoch,
        "threshold": best_threshold,
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_auc": test_metrics["auc"],
        "history_path": history_path,
        "predictions_path": predictions_path,
        "confusion_matrix_path": cm_path
    }


    del model
    torch.cuda.empty_cache()
    gc.collect()

    return result_row



all_results = []

for cfg in model_configs:
    try:
        result = train_one_model(
            model_key=cfg["model_key"],
            model_name=cfg["model_name"],
            family=cfg["family"]
        )

        all_results.append(result)

        intermediate_df = pd.DataFrame(all_results)
        intermediate_path = os.path.join(OUTPUT_DIR, "intermediate_results.csv")
        intermediate_df.to_csv(intermediate_path, index=False)

    except Exception as e:
        print("\nERROR with model:", cfg["model_key"])
        print(e)

results_df = pd.DataFrame(all_results)

results_path = os.path.join(OUTPUT_DIR, "cnn_vs_vit_comparison_results.csv")
results_df.to_csv(results_path, index=False)

print("\n=== FINAL CNN vs ViT RESULTS ===")
display(results_df)

print("Saved final results:", results_path)



ranking_df = results_df.sort_values("test_auc", ascending=False).reset_index(drop=True)
ranking_df["rank_by_auc"] = np.arange(1, len(ranking_df) + 1)

ranking_path = os.path.join(OUTPUT_DIR, "cnn_vs_vit_ranking_by_auc.csv")
ranking_df.to_csv(ranking_path, index=False)

print("\n=== RANKING BY AUC ===")
display(ranking_df)



family_summary = results_df.groupby("family")[
    ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_auc"]
].agg(["mean", "std"])

family_summary_path = os.path.join(OUTPUT_DIR, "family_summary.csv")
family_summary.to_csv(family_summary_path)

print("\n=== FAMILY SUMMARY ===")
display(family_summary)



plot_df = results_df.copy()
plot_df = plot_df.sort_values("test_auc", ascending=False)

metrics_to_plot = ["test_accuracy", "test_f1", "test_auc"]

plt.figure(figsize=(11, 6))

x = np.arange(len(plot_df))
width = 0.25

plt.bar(x - width, plot_df["test_accuracy"] * 100, width, label="Accuracy")
plt.bar(x, plot_df["test_f1"] * 100, width, label="F1")
plt.bar(x + width, plot_df["test_auc"] * 100, width, label="AUC")

plt.xticks(x, plot_df["model_key"], rotation=35, ha="right")
plt.ylabel("Score (%)")
plt.title("CNN vs Vision Transformer Comparison on ISIC 2019 Binary Task")
plt.legend()
plt.grid(axis="y")

barplot_path = os.path.join(OUTPUT_DIR, "cnn_vs_vit_barplot.png")
plt.savefig(barplot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved barplot:", barplot_path)



family_plot_df = results_df.groupby("family")[["test_accuracy", "test_f1", "test_auc"]].mean().reset_index()

plt.figure(figsize=(8, 5))

x = np.arange(len(family_plot_df))
width = 0.25

plt.bar(x - width, family_plot_df["test_accuracy"] * 100, width, label="Accuracy")
plt.bar(x, family_plot_df["test_f1"] * 100, width, label="F1")
plt.bar(x + width, family_plot_df["test_auc"] * 100, width, label="AUC")

plt.xticks(x, family_plot_df["family"], rotation=20, ha="right")
plt.ylabel("Mean Score (%)")
plt.title("Average Performance by Model Family")
plt.legend()
plt.grid(axis="y")

family_plot_path = os.path.join(OUTPUT_DIR, "family_comparison_barplot.png")
plt.savefig(family_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved family plot:", family_plot_path)



print("\nExperiment completed.")
print("Output directory:", OUTPUT_DIR)
print("Results:", results_path)
print("Ranking:", ranking_path)
print("Family summary:", family_summary_path)
print("Barplot:", barplot_path)
print("Family plot:", family_plot_path)